In [0]:
import os
from pyspark.sql import functions as F
from pyspark.sql.streaming import StreamingQueryException

current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

landing_path = f"{base_path}/landing"
bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
schema_location = f"{base_path}/bronze/_schema"
checkpoint_location = f"{base_path}/bronze/_checkpoint"

In [0]:
def run_batch_ingestion(source_path, max_retries=3):
    """Ingests a single landing directory with self-healing schema evolution handling."""
    for attempt in range(1, max_retries + 1):
        try:
            df_bronze_stream = (
                spark.readStream.format("cloudFiles")
                .option("cloudFiles.format", "json")
                .option("cloudFiles.schemaLocation", schema_location)
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.rescuedDataColumn", "_rescued_data")
                .load(source_path)
                .withColumn("_ingested_at", F.current_timestamp())
                .withColumn("_source_file", F.col("_metadata.file_path"))
            )

            query = (
                df_bronze_stream.writeStream
                .format("delta")
                .outputMode("append")
                .option("checkpointLocation", checkpoint_location)
                .option("mergeSchema", "true")
                .trigger(availableNow=True)
                .start(bronze_table_path)
            )

            query.awaitTermination()
            break

        except StreamingQueryException as sqe:
            error_str = str(sqe)
            if "UnknownFieldException" in error_str or "NEW_FIELDS_IN_RECORD" in error_str or "DELTA_METADATA_MISMATCH" in error_str:
                print(f"  [SCHEMA EVOLUTION] Detected new fields. Self-healing stream restart...")
                if attempt == max_retries:
                    raise sqe
            else:
                raise sqe


# Automatically discover and process all batch directories sequentially
batch_folders = sorted([f.name.rstrip('/') for f in dbutils.fs.ls(landing_path) if "batch_" in f.name])

print(f"Found {len(batch_folders)} batch directories to process: {batch_folders}\n")

for batch_name in batch_folders:
    print(f"=== PROCESSING: {batch_name} ===")
    batch_source_path = f"{landing_path}/{batch_name}"
    run_batch_ingestion(batch_source_path)
    print(f"Successfully processed {batch_name}.\n")

print("All batches ingested into Bronze automatically.")

In [0]:
current_user = spark.sql("SELECT current_user()").collect()[0][0]
default_workspace_path = f"/Workspace/Users/{current_user}/delta_assignment"

dbutils.widgets.text("base_path", default_workspace_path, "Base Workspace Path")
base_path = dbutils.widgets.get("base_path")

bronze_table_path = f"{base_path}/bronze/earthquakes_bronze"
df_bronze = spark.read.format("delta").load(bronze_table_path)

print("=== AUTOMATED BRONZE INGESTION VERIFICATION ===")
print(f"Total Bronze Records: {df_bronze.count()} (Expected: 210)")
print("Table Schema Columns:", df_bronze.columns)

rescued_records = df_bronze.filter(F.col("_rescued_data").isNotNull())
print(f"Captured Rescued Data Records: {rescued_records.count()} (Expected: 10)")